
BRONZE PROCESSING

In [0]:
%run /Workspace/Users/malni1880@gmail.com/FMCG_project/1_setup/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable 

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
catalog=dbutils.widgets.get("catalog")
data_source=dbutils.widgets.get("data_source")

print(catalog,data_source)

In [0]:
base_path=f'/Volumes/fmcg/source_data/raw/{data_source}.csv'
print(base_path)

In [0]:
df= (
    spark.read.format("csv")
        .option("header",True)
        .option("inferSchema",True)
        .load(base_path)
        .withColumn("read_timestamp",F.current_timestamp())
        .select("*","_metadata.file_name","_metadata.file_size")
    )
    
df.display(10)

In [0]:
df.printSchema()

In [0]:
df.write.mode("overwrite") \
        .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
        .saveAsTable(f'{catalog}.{bronze_schema}.{data_source}')


SILVER PROCESSING

In [0]:
df_bronze=spark.sql(f'select * from fmcg.bronze.customers')

df_bronze.display()

In [0]:
df_bronze.groupBy("customer_id").count().where(F.col("count")>1).display()

In [0]:
print("Rows before duplicates drop: ",df_bronze.count())
df_silver=df_bronze.dropDuplicates(["customer_id"])
print("Rows after duplicates drop: ",df_silver.count())

In [0]:
df_silver=(
    df_silver.withColumn("customer_name",F.trim(F.col("customer_name")))
)

In [0]:
display(df_silver.filter(F.col("customer_name")!= F.trim(F.col("customer_name"))))

In [0]:
df_silver.select("city").distinct().show()

In [0]:
city_mapping={
    "Bangalore":"Bengaluru",
    "Bengalore":"Bengaluru",

    "Hyderabadd":"Hyderabad",
    "Hyderbad":"Hyderabad",

    "NewDelhee":"New Delhi",
    "NewDheli":"New Delhi",
    "NewDelhi":"New Delhi"
}

allowed=["Hyderabad","Bengaluru","New Delhi"]

df_silver = df_silver.replace(city_mapping, subset="city")\
    .withColumn("city",F.when(F.col("city").isNull(),None)
                .when(F.col("city").isin(allowed),F.col("city"))
                .otherwise(None)
                )






In [0]:
df_silver.filter(F.col("city").isNull()).show(truncate=False)

In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver=df_silver.withColumn("customer_name", F.when(F.col("customer_name").isNull(),None)
                .otherwise(F.initcap(F.col("customer_name"))))

df_silver.select("customer_name").distinct().show()

In [0]:
df_silver.filter(F.col("city").isNull())\
    .select("customer_name").distinct()\
    .show(truncate=False)

In [0]:
null_customer_name=['Endurance Foods','Macrobite Superfoods','Sprintx Nutrition','Zenathlete Foods','Peak Performance Store','Primefuel Nutrition','Recovery Lane']

df_silver.filter(F.col("customer_name").isin(null_customer_name)).display()
    

In [0]:
customer_city_fix={
    789101: "Bengaluru",
    789221: "Hyderabad",
    789403: "New Delhi",
    789420: "New Delhi",
    789421: "Bengaluru",
    789422: "Hyderabad",
    789503:"Hyderabad",
    789520: "New Delhi",
    789521: "Bengaluru",
    789522: "Hyderabad",
    789603: "New Delhi"
}

df_fix= spark.createDataFrame(customer_city_fix.items(),["customer_id","fixed_city"])
display(df_fix)

In [0]:
df_silver=df_silver.join(df_fix, on="customer_id", how="left")\
    .withColumn("city",F.coalesce("city","fixed_city"))\
    .drop("fixed_city")

In [0]:
display(df_silver)

In [0]:
display(df_silver.filter(F.col("city").isNull()))

In [0]:
df_silver=df_silver.withColumn("customer_id", F.col("customer_id").cast("string"))
df_silver.printSchema()

In [0]:
df_silver=df_silver.withColumn("customer",F.concat_ws("-",F.col("customer_name"),F.coalesce(F.col("city"),F.lit("Unknown"))))\
    .withColumn("market",F.lit("India"))\
    .withColumn("platform",F.lit("Sports Bar"))\
    .withColumn("channel",F.lit("Acquisition"))


In [0]:
display(df_silver)

In [0]:
df_silver.write.format("delta").mode("overwrite")\
    .option("delta.enableChangeDataFeed", "true")\
    .option("mergeSchema", "true")\
    .saveAsTable("fmcg.silver.customers")

### GOLD PROCESSING

In [0]:
df_silver=spark.sql("select * from fmcg.silver.customers")

df_gold=df_silver.select("customer_id","customer","customer_name","city","market","platform","channel")
display(df_gold)

In [0]:
df_gold.write.format("delta").mode("overwrite")\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable("fmcg.gold.sb_dim_customers")

In [0]:
delta_table=DeltaTable.forName(spark,"fmcg.gold.dim_customers")
df_child_customer=spark.table("fmcg.gold.sb_dim_customers").select(F.col("customer_id").alias("customer_code"),"customer","platform","market","channel")

In [0]:
delta_table.alias("target").merge(source=df_child_customer.alias("source"),condition=F.expr("target.customer_code=source.customer_code"))\
.whenMatchedUpdateAll()\
.whenNotMatchedInsertAll().execute()